# ಪಾಠ 11 - ಏಜೆಂಟ್‌ನಿಂದ ಏಜೆಂಟ್ (A2A) ಪ್ರೋಟೋಕಾಲ್


## ಸೆಟಪ್


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## A2A ಪ್ರೋಟೋಕಾಲ್ ಎಂದರೆ ಏನು?

ಈ **ಏಜೆಂಟ್-ರಿಂದ-ಏಜೆಂಟ್ (A2A) ಪ್ರೋಟೋಕಾಲ್** ಒಂದು ತೆರೆಯಾದ ಮಾನದಂಡವಾಗಿದೆ, ಅದು ಎಐ ಏಜೆಂಟ್‌ಗಳು ಪರಸ್ಪರ ಸಂವಹನ ಮಾಡಲು,
ಒಬ್ಬರನ್ನೊಬ್ಬರನ್ನು ಕಂಡುಹಿಡಿಯಲು ಮತ್ತು ಸಹಕರಿಸಲು — ಅವುಗಳು ವಿಭಿನ್ನ ಫ್ರೇಮ್ವರ್ಕ್‌ಗಳಲ್ಲಿ ನಿರ್ಮಿಸಲ್ಪಟ್ಟಿದ್ದರೂ ಅಥವಾ
ವಿಭಿನ್ನ ಸೇವೆಗಳಲ್ಲಿ ಹೋಸ್ಟ್ ಮಾಡಲ್ಪಟ್ಟಿದ್ದರೂ ಸಹ.

ಪ್ರಮುಖ ಕಲ್ಪನೆಗಳು:

- **ಶೋಧನೆ** – ಏಜೆಂಟ್‌ಗಳು ಅವರ ಸಾಮರ್ಥ್ಯಗಳನ್ನು ವಿವರಿಸುವ *Agent Card* ಅನ್ನು ಪ್ರಕಟಿಸುತ್ತವೆ, ಇದರಿಂದ
  ಇತರ ಏಜೆಂಟ್‌ಗಳು (ಅಥವಾ ಒರ್ಕೆಸ್ಟ್ರೇಟರ್‌ಗಳು) ಕಾರ್ಯಕ್ಕೆ ಸೂಕ್ತ ತಜ್ಞರನ್ನು ಸುಲಭವಾಗಿ ಕಂಡುಹಿಡಿಯಬಹುದು.
- **ಸಂದೇಶ ವಿನಿಮಯ** – ಏಜೆಂಟ್‌ಗಳು ಸಾಮಾನ್ಯ ಪ್ರೋಟೋಕಾಲ್ ಮೂಲಕ ರಚನಾತ್ಮಕ ಸಂದೇಶಗಳನ್ನು ವಿನಿಮಯ ಮಾಡುತ್ತವೆ, ಆದ್ದರಿಂದ ಒಂದು
  ಏಜೆಂಟ್‌ನಿಂದ ಬಂದ ವಿನಂತಿಯನ್ನು ಮತ್ತೊಂದು ಏಜೆಂಟ್ ಅರ್ಥಮಾಡಿಕೊಂಡು ಮತ್ತು ಪೂರೈಸಬಲ್ಲದು, ಅದರ ಆಂತರಿಕ
  ಅನುಷ್ಠಾನಕ್ಕೆ ಸಂಬಂಧವಿಲ್ಲದೆ.
- **ಕಾರ್ಯ ಜೀವನಚಕ್ರ** – A2A ಕೆಳಗಿನಂತಹ ಸ್ಥಿತಿಗಳನ್ನು ವ್ಯಾಖ್ಯಾನಿಸುತ್ತದೆ: *ಸಲ್ಲಿಸಲಾಗಿದೆ*, *ಕಾರ್ಯನಿರ್ವಹಿಸಲಾಗುತ್ತಿದೆ*, *ಸಂಪೂರ್ಣವಾಗಿದೆ*, ಮತ್ತು
  *ವಿಫಲವಾಗಿದೆ*, ಇದು ಒರ್ಕೆಸ್ಟ್ರೇಟರ್‌ಗೆ ನಿಯೋಜಿಸಲಾದ ಕಾರ್ಯವು ಹೇಗೆ ಪ್ರಗತಿಯಾಗುತ್ತಿದೆ ಎಂಬುದರ ಸಂಪೂರ್ಣ ದೃಷ್ಠಿಯನ್ನು ನೀಡುತ್ತದೆ.

ಈ ಪಾಠದಲ್ಲಿ ನಾವು A2A-ಶೈಲಿ ಸಹಕಾರವನ್ನು ಅನುಕರಿಸುತ್ತೇವೆ, ಮೂರು ವಿಶೇಷೀಕೃತ ಪ್ರವಾಸ ಏಜೆಂಟ್‌ಗಳನ್ನು
ಒಂದು ಕಾರ್ಯಪ್ರವಾಹಕ್ಕೆ ಜೋಡಿಸಿ, ಪ್ರತಿಯೊಂದು ಏಜೆಂಟ್ ತನ್ನ ಪರಿಣತಿಯನ್ನು ನೀಡುತ್ತದೆ ಮತ್ತು ಫಲಿತಾಂಶಗಳನ್ನು ಮುಂದಿನವರಿಗೆ ಹಂಚುತ್ತದೆ.


## ವಿಶೇಷೀಕೃತ ಪ್ರಯಾಣ ಏಜೆಂಟ್‌ಗಳನ್ನು ರಚಿಸುವುದು


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## ಕಾರ್ಯಪ್ರವಾಹದ ಮೂಲಕ ಬಹು-ಏಜೆಂಟ್ ಸಹಕಾರ

ನಾವು ಮೂರು ಏಜೆಂಟ್ಗಳನ್ನು ಕ್ರಮಾನುಗತ ಕಾರ್ಯಪ್ರವಾಹಕ್ಕೆ ಸಂಪರ್ಕಿಸುತ್ತೇವೆ, ಇದು A2A ಸಂದೇಶ ವಹಿವಾಟನ್ನು ಪ್ರತಿಬಿಂಬಿಸುತ್ತದೆ:

1. **CurrencyExchangeAgent** ಬಳಕೆದಾರರ ವಿನಂತಿಯನ್ನು ಸ್ವೀಕರಿಸುತ್ತದೆ ಮತ್ತು ಕರೆನ್ಸಿ ಮಾರ್ಗದರ್ಶನವನ್ನು ಒದಗಿಸುತ್ತದೆ.
2. **ActivityPlannerAgent** ವೃದ್ಧಿಗೊಂಡ ಹಿನ್ನೆಲೆಯನ್ನು ಸ್ವೀಕರಿಸಿ ಚಟುವಟಿಕೆ ಶಿಫಾರಸುಗಳನ್ನು ಸೇರಿಸುತ್ತದೆ.
3. **TravelManagerAgent** ಎರಡೂ ಇನ್ಪುಟ್‌ಗಳನ್ನು ಸಂಶ್ಲೇಷಿಸಿ ಅಂತಿಮ ಪ್ರಯಾಣ ಸಾರಾಂಶವನ್ನು ರಚಿಸುತ್ತದೆ.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ಉತ್ಪಾದನೆಯಲ್ಲಿ A2A ಅನ್ನು ಅರ್ಥಮಾಡಿಕೊಳ್ಳುವುದು

ಉತ್ಪಾದನಾ ಪರಿಸರದಲ್ಲಿ A2A ಪ್ರೋಟೋಕಾಲ್ ಶಕ್ತಿಶಾಲಿ ಕ್ರಾಸ್-ಸರ್ವಿಸ್ ಸನ್ನಿವೇಶಗಳನ್ನು ಮುಕ್ತಗೊಳಿಸುತ್ತದೆ:

| ಸಾಮರ್ಥ್ಯ | ವಿವರಣೆ |
|---|---|
| **ಫ್ರೇಮ್‌ವರ್ಕ್‌ಗಳ ನಡುವಿನ ಪರಸ್ಪರ ಕಾರ್ಯಗತಗೊಳಿಸುವಿಕೆ** | ಒಂದು ಫ್ರೇಮ್‌ವರ್ಕ್‌ನಲ್ಲಿ ನಿರ್ಮಿತ ಏಜೆಂಟ್ ಯಾವುದೇ ಇನ್ನೊಬ್ಬ A2A-compliant ಫ್ರೇಮ್‌ವರ್ಕ್‌ನಲ್ಲಿ ನಿರ್ಮಿತ ಏಜೆಂಟ್‌ಗೆ ಕಾರ್ಯಗಳನ್ನು ಹಂಚಿಕೆ ಮಾಡಬಹುದು, ಇದರಿಂದ ನಿಜವಾದ ಸಂಸ್ಥಾ-ಅಂತರ ಪರಸ್ಪರ ಕಾರ್ಯಸಾಧ್ಯತೆ ಸಾಧ್ಯವಾಗುತ್ತದೆ. |
| **ಸೇವಾ ಸೀಮೆಗಳು** | ಏಜೆಂಟ್ಗಳು ಪ್ರತ್ಯೇಕ ಮೈಕ್ರೋಸರ್ವಿಸಸ್‌ಗಳಲ್ಲ, ಕ್ಲೌಡ್ ಪ್ರಾಂತ್ಯಗಳಲ್ಲಿ ಅಥವಾ ವಿಭಿನ್ನ ಸಂಸ್ಥೆಗಳಲ್ಲಿ ಇರಬಹುದು, ಮತ್ತು ಆಗಲೂ ಸುಗಮವಾಗಿ ಸಹಕರಿಸಬಹುದು. |
| **ಡೈನಾಮಿಕ್ ಕಂಡುಹಿಡಿತ** | ಒর্কೆಸ್ಟ್ರೇಟರ್ ಕಾರ್ಯನಿರ್ವಹಣಾ ಸಮಯದಲ್ಲಿ Agent Card ರೆಜಿಸ್ಟ್ರಿಯನ್ನು ಪ್ರಶ್ನಿಸಿ ನೀಡಿರುವ ಉಪ-ಕಾರ್ಯದಕ್ಕೆ ಸೂಕ್ತವಾದ ತಜ್ಞನನ್ನು ಕಂಡುಹಿಡಿಯಬಹುದು. |
| **ಸ್ಟ್ರೀಮಿಂಗ್ & ಪುಶ್ ನೋಟಿಫಿಕೇಶನ್ಗಳು** | A2A ವಾಸ್ತುಕಾಲೀನ ಪ್ರಗತಿ ಅಪ್‌ಡೇಟ್‌ಗಳಿಗಾಗಿ Server-Sent Events (SSE) ಹಾಗೂ ದೀರ್ಘಕಾಲ ಕಾರ್ಯಗಳಿಗೆ ಪುಶ್ ನೋಟಿಫಿಕೇಶನ್ಗಳನ್ನು ಬೆಂಬಲಿಸುತ್ತದೆ. |

ಮೇಲಿನ ನಾವು ನಿರ್ಮಿಸಿದ ವರ್ಕ್‌ಫ್ಲೋ ಈ ಮಾದರಿಯ ಒಂದು ಸರಳಗೊಳಿಸಿದ, ಪ್ರಕ್ರಿಯೆಯೊಳಗಿನ ಆವೃತ್ತಿಯಾಗಿದೆ. ವಾಸ್ತವಿಕ ನಿಯೋಜನೆಯಲ್ಲಿ ಪ್ರತಿ ಏಜೆಂಟ್ ಒಂದು HTTP ಎಂಡ್ಪಾಯಿಂಟ್ ಹೊರಗೊಳಿಸುತ್ತದೆ, Agent Card ಅನ್ನು ಪ್ರಕಟಿಸುತ್ತದೆ ಮತ್ತು A2A JSON-RPC ಪ್ರೋಟೋಕಾಲ್ ಮೂಲಕ ಸಂವಹನ ಮಾಡುತ್ತದೆ.


## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ತಿಳಿದಿರುವುದು:

1. **A2A ಪ್ರೋಟೋಕಾಲ್ ಎಂದರೆ ಏನು** — ಏಜೆಂಟ್‌-ಮಧ್ಯೆ ಅನ್ವೇಷಣೆ, ಸಂದೇಶ ವಿನಿಮಯ,
   ಮತ್ತು ಕಾರ್ಯ ನಿರ್ವಹಣೆ.
2. **ವಿಶೇಷ ಏಜೆಂಟ್‌ಗಳನ್ನು ಹೇಗೆ ರಚಿಸುವುದು** — ಒಂದು ಕರೆನ್ಸಿ ವಿನಿಮಯ ಏಜೆಂಟ್, ಒಂದು ಚಟುವಟಿಕೆ ಯೋಜಕ ಏಜೆಂಟ್,
   ಮತ್ತು ಒಂದು ಪ್ರವಾಸ ನಿರ್ವಾಹಕ ಸಮನ್ವಯಕ.
3. **ಏಜೆಂಟ್‌ಗಳನ್ನು ವರ್ಕ್‌ಫ್ಲೋಗೆ ಜೋಡಿಸುವುದು ಹೇಗೆ** — `WorkflowBuilder` ಬಳಸಿ ಕ್ರಮಬದ್ಧ
   ಏಜೆಂಟ್‌ಗಳ ನಡುವೆ ಸಂದೇಶ ಹಂಚಿಕೆಯನ್ನು ಮಾದರಿಪಡಿಸಲು.
4. **ಉತ್ಪಾದನೆಯಲ್ಲಿ A2A ಹೇಗೆ ಕಾರ್ಯನಿರ್ವಹಿಸುತ್ತದೆ** — ಫ್ರೇಮ್‌ವರ್ಕ್‌ಗಳ ಮತ್ತು ಸೇವೆಗಳ ನಡುವೆ ಅಡ್ಡ-ಸಹಕಾರವನ್ನು ಸಕ್ರಿಯಗೊಳಿಸುವುದು
   ಡೈನಾಮಿಕ್ ಅನ್ವೇಷಣೆ ಮತ್ತು ಸ್ಟ್ರೀಮಿಂಗ್ ಅಪ್ಡೇಟ್‌ಗಳೊಂದಿಗೆ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
ಅಸ್ವೀಕರಣ:
ಈ ದಾಖಲೆ AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲ್ಪಟ್ಟಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಶುದ್ಧತೆಗಳಿರಬಹುದು ಎಂಬುದನ್ನು ದಯವಿಟ್ಟು ಗಮನದಲ್ಲಿಟ್ಟುಕೊಳ್ಳಿ. ಮೂಲ ದಾಖಲೆ ಅದರ ಸ್ವಭಾಷೆಯಲ್ಲಿ ಅಧಿಕೃತ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಗಂಭೀರ ಅಥವಾ ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದದ ಬಳಕೆಯಿಂದ ಉಂಟಾಗುವ ಯಾವುದೇ ಗೊಂದಲಗಳು ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳಿಗಾಗಿ ನಾವು ಜವಾಬ್ದಾರರಾಗುವುದಿಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
